# F.E. Walter prompt comparison
Check the Rule-based baseline and both LLM cases, With and Without operational knowledge, without contacting a model provider, then inspect the prompts used by the two LLM configurations.

In [ ]:
from pathlib import Path
import csv, json, subprocess, sys, yaml

root = Path.cwd()
if not (root / 'examples' / 'fe_walter' / 'seasonal_target.yaml').exists():
    root = root.parent
example_dir = root / 'examples' / 'fe_walter'
config_paths = {
    'Rule-based': example_dir / 'rule_based.yaml',
    'Without operational knowledge': example_dir / 'seasonal_target.yaml',
    'With operational knowledge': example_dir / 'dkp.yaml',
}
configs = {name: yaml.safe_load(path.read_text(encoding='utf-8')) for name, path in config_paths.items()}
case = json.loads((example_dir / 'case.json').read_text(encoding='utf-8'))
inflow_path = example_dir / case['inflow_source']['path']
with inflow_path.open(newline='', encoding='utf-8') as stream:
    dates = sorted({row['date'] for row in csv.DictReader(stream)})
case_summary = {
    'input_days': len(dates),
    'input_period': [dates[0], dates[-1]],
    'tutorial_windows': {name: [cfg['start_date'], cfg['end_date']] for name, cfg in configs.items()},
    'reservoirs': list(case['network']['reservoirs']),
}
case_summary

## Inspect the selected prompts
All three configurations use the first seven days of the shared F.E. Walter study inflows and the same initial storage and routing. The YAML files select the public SI-unit prompts, and With operational knowledge includes local operating guidance.

In [ ]:
prompt_texts = {}
comparison = {}
for name, config in configs.items():
    agent = config['agents']['walter_operator']
    if agent['mode'] == 'offline':
        continue
    selected = {kind: example_dir / relative for kind, relative in agent['prompt'].items()}
    prompt_texts[name] = {kind: path.read_text(encoding='utf-8') for kind, path in selected.items()}
    comparison[name] = {
        'system_prompt': selected['system'].relative_to(root).as_posix(),
        'user_prompt': selected['user'].relative_to(root).as_posix(),
        'knowledge': agent.get('knowledge', {}),
        'model': config['models']['local']['model'],
        'base_url': config['models']['local']['base_url'],
    }
comparison

## Validate all three configurations offline
`check` resolves the case, prompt, knowledge, and model settings. It does not contact Ollama and does not create run output.

In [ ]:
command = [sys.executable, '-m', 'llm_reservoir_agent.workflow', 'check']
check_results = {}
for name, path in config_paths.items():
    completed = subprocess.run(command + [str(path)], text=True, capture_output=True)
    if completed.returncode:
        raise RuntimeError(completed.stderr or completed.stdout)
    check_results[name] = json.loads(completed.stdout)
check_summary = {
    name: {
        'schema_version': result['schema_version'],
        'days': result['days'],
        'network_calls': result['network_calls'],
        'live_ready': result['live_ready'],
    }
    for name, result in check_results.items()
}
check_summary

## Optional runs
The next cell prints commands only. Rule-based runs offline. For either LLM case, review the prompts, start Ollama with `gemma4:12b`, choose fresh `run_id` and `output` values, and then run a command manually. `--allow-network` explicitly authorizes requests to the configured local endpoint.

In [ ]:
opt_in_commands = [
    'reservoir-experiment run examples/fe_walter/rule_based.yaml',
    'reservoir-experiment run examples/fe_walter/seasonal_target.yaml --allow-network',
    'reservoir-experiment run examples/fe_walter/dkp.yaml --allow-network',
]
print('\n'.join(opt_in_commands))

## Next steps
Read the [F.E. Walter guide](../examples/fe_walter/README.md) for verification and output paths. Use the [results guide](../docs/RESULTS_GUIDE.md) to interpret decisions, provider attempts, physical trajectories, and provenance.